<a href="https://colab.research.google.com/github/njwbilll/Tugas-5_Grokking-Deep-Learning-MANNING_Najwa-Bilqis-Al-Khalidah/blob/main/13_Introducing_Automatic_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13: Memperkenalkan Optimasi Otomatis Mari Membangun Kerangka Kerja Pembelajaran Mendalam

**Referensi:** Grokking Deep Learning, Andrew W. Trask (Manning Publications, 2019)

## Ringkasan Bab

Bab ketiga belas ini menjadi tonggak sejarah yang sangat monumental dalam buku ini. Sepanjang dua belas bab sebelumnya, kita selalu merumuskan dan menuliskan fungsi turunan kalkulus secara manual setiap kali kita menambah lapisan baru atau mengubah arsitektur matematika. Pendekatan primitif tersebut tentu sangat menyiksa dan rawan terjadi kesalahan operasional. Bab ini membedah rahasia di balik piranti lunak raksasa industri seperti PyTorch maupun TensorFlow. Kita diajak untuk merancang sebuah struktur data pintar bernama Tensor yang secara mandiri mampu melacak riwayat komputasinya sendiri dan merambat balikkan aliran kesalahan secara otomatis.

## Pemahaman Teori Grafik Komputasi dan Diferensiasi Otomatis

### A. Bencana Kalkulasi Manual
Bayangkan kita memiliki arsitektur jaringan saraf dengan belasan lapisan yang dipenuhi dengan operasi logaritma, eksponensial, dan trigonometri yang saling bersilangan. Menyusun rumus turunan manual untuk skenario serumit ini adalah sebuah rintangan mematikan. Otomatisasi sangat mutlak dibutuhkan agar peneliti cukup memikirkan desain aliran maju tanpa perlu memusingkan lintasan penjalaran balik.

### B. Konsep Entitas Cerdas Tensor
Sebuah Tensor tidak hanya sekadar wadah mati untuk menyimpan daftar angka matriks. Ia adalah sebuah entitas komputasi cerdas yang menyimpan tiga properti sekaligus, yakni nilai datanya sendiri, nilai gradien pelatihannya, dan pointer referensi yang menunjuk kepada siapa entitas lain yang berpartisipasi menciptakan dirinya. Rangkaian saling tunjuk ini kelak secara otomatis membentuk sebuah silsilah keluarga matematis yang dikenal sebagai Grafik Komputasi.

### C. Propagasi Mundur Terpimpin Mandiri
Ketika kita memicu sinyal mundur secara eksplisit pada ujung garis finis perhitungan kerugian, sinyal tersebut akan mengalir mencari penciptanya secara berantai, mendelegasikan pecahan turunan sesuai operasi matematika yang menyambungkan mereka. Entitas perkalian akan menyilangkan gradien, entitas penjumlahan akan mendistribusikan gradien secara merata, dan entitas matriks akan menyelaraskan bidang transposisi secara instan.

### Tahap Persiapan Pustaka Utilitas
Kita memuat modul operasional aljabar untuk mengatur pembentukan struktur kerangka kerja otonom yang menggantikan perumusan turunan manual.

In [1]:
import numpy as np
print("Modul aljabar dasar siap untuk direkonstruksi menjadi kerangka kerja diferensiasi otomatis")

Modul aljabar dasar siap untuk direkonstruksi menjadi kerangka kerja diferensiasi otomatis


### Konstruksi Kelas Kerangka Tensor Paripurna
Di blok kode pamungkas ini, kita menciptakan ulang logika inti dari sebuah kerangka kerja modern. Objek ini dipersenjatai dengan kemampuan manipulasi operator matematika bawaan dari dasar bahasa pemrograman, sehingga saat dua belah objek ditambahkan, mereka secara rahasia mencatat jejak kolaborasi tersebut demi kepentingan perambatan galat di masa depan.

In [2]:
class KerangkaTensor:
    def __init__(self, matriks_data, optimasi_otomatis=False, daftar_pembuat=None, operasi_sumber=None, kode_identitas=None):
        self.data = np.array(matriks_data)
        self.optimasi_otomatis = optimasi_otomatis
        self.gradien = None

        if kode_identitas is None:
            self.kode_identitas = np.random.randint(0, 1000000)
        else:
            self.kode_identitas = kode_identitas

        self.daftar_pembuat = daftar_pembuat
        self.operasi_sumber = operasi_sumber
        self.katalog_anak = {}

        if daftar_pembuat is not None:
            for entitas_pembuat in daftar_pembuat:
                if self.kode_identitas not in entitas_pembuat.katalog_anak:
                    entitas_pembuat.katalog_anak[self.kode_identitas] = 1
                else:
                    entitas_pembuat.katalog_anak[self.kode_identitas] = np.add(entitas_pembuat.katalog_anak[self.kode_identitas], 1)

    def status_gradien_anak_terpenuhi(self):
        for identitas_anak, jumlah_tunggu in self.katalog_anak.items():
            if jumlah_tunggu != 0:
                return False
        return True

    def mundur(self, input_gradien=None, asal_usul_gradien=None):
        if self.optimasi_otomatis:
            if input_gradien is None:
                input_gradien = KerangkaTensor(np.ones_like(self.data))

            if asal_usul_gradien is not None:
                if self.katalog_anak[asal_usul_gradien.kode_identitas] == 0:
                    raise Exception("Aliran propagasi mundur menembus batas maksimal ekspektasi grafik")
                else:
                    self.katalog_anak[asal_usul_gradien.kode_identitas] = np.subtract(self.katalog_anak[asal_usul_gradien.kode_identitas], 1)

            if self.gradien is None:
                self.gradien = input_gradien
            else:
                self.gradien = self.gradien + input_gradien

            if self.daftar_pembuat is not None and (self.status_gradien_anak_terpenuhi() or asal_usul_gradien is None):
                if self.operasi_sumber == "penjumlahan":
                    self.daftar_pembuat[0].mundur(self.gradien, self)
                    self.daftar_pembuat[1].mundur(self.gradien, self)

                if self.operasi_sumber == "negasi":
                    self.daftar_pembuat[0].mundur(self.gradien.__neg__(), self)

                if self.operasi_sumber == "perkalian":
                    gradien_jalur_pertama = self.gradien * self.daftar_pembuat[1]
                    self.daftar_pembuat[0].mundur(gradien_jalur_pertama, self)

                    gradien_jalur_kedua = self.gradien * self.daftar_pembuat[0]
                    self.daftar_pembuat[1].mundur(gradien_jalur_kedua, self)

                if self.operasi_sumber == "perkalian_matriks":
                    turunan_kiri = self.gradien.eksekusi_dot(self.daftar_pembuat[1].transposisi())
                    self.daftar_pembuat[0].mundur(turunan_kiri, self)

                    turunan_kanan = self.daftar_pembuat[0].transposisi().eksekusi_dot(self.gradien)
                    self.daftar_pembuat[1].mundur(turunan_kanan, self)

                if self.operasi_sumber == "transposisi":
                    self.daftar_pembuat[0].mundur(self.gradien.transposisi(), self)

    def __add__(self, entitas_lain):
        if self.optimasi_otomatis and entitas_lain.optimasi_otomatis:
            return KerangkaTensor(np.add(self.data, entitas_lain.data),
                                  optimasi_otomatis=True,
                                  daftar_pembuat=[self, entitas_lain],
                                  operasi_sumber="penjumlahan")
        return KerangkaTensor(np.add(self.data, entitas_lain.data))

    def __neg__(self):
        if self.optimasi_otomatis:
            return KerangkaTensor(np.subtract(0.0, self.data),
                                  optimasi_otomatis=True,
                                  daftar_pembuat=[self],
                                  operasi_sumber="negasi")
        return KerangkaTensor(np.subtract(0.0, self.data))

    def __sub__(self, entitas_lain):
        return self + entitas_lain.__neg__()

    def __mul__(self, entitas_lain):
        if self.optimasi_otomatis and entitas_lain.optimasi_otomatis:
            return KerangkaTensor(np.multiply(self.data, entitas_lain.data),
                                  optimasi_otomatis=True,
                                  daftar_pembuat=[self, entitas_lain],
                                  operasi_sumber="perkalian")
        return KerangkaTensor(np.multiply(self.data, entitas_lain.data))

    def eksekusi_dot(self, entitas_lain):
        if self.optimasi_otomatis and entitas_lain.optimasi_otomatis:
            return KerangkaTensor(np.dot(self.data, entitas_lain.data),
                                  optimasi_otomatis=True,
                                  daftar_pembuat=[self, entitas_lain],
                                  operasi_sumber="perkalian_matriks")
        return KerangkaTensor(np.dot(self.data, entitas_lain.data))

    def transposisi(self):
        if self.optimasi_otomatis:
            return KerangkaTensor(self.data.transpose(),
                                  optimasi_otomatis=True,
                                  daftar_pembuat=[self],
                                  operasi_sumber="transposisi")
        return KerangkaTensor(self.data.transpose())

    def __repr__(self):
        return str(self.data.__repr__())

print("Arsitektur dasar diferensiasi otomatis otonom berhasil dikompilasi secara utuh")

Arsitektur dasar diferensiasi otomatis otonom berhasil dikompilasi secara utuh


### Mekanisme Pengoptimalan Otomatis
Jika pekerjaan perambatan balik kini ditangani sepenuhnya oleh entitas Tensor, maka urusan pembaruan parameter dikerjakan oleh modul Penurunan Gradien Stokastik independen. Modul pengawas ini bertugas berkeliling menyapu seluruh blok memori internal jaringan, mereduksi nilai gradien menggunakan sebuah rasio pembelajaran, lalu mengurangkannya dari nilai memori asli.

In [3]:
class PenurunanGradienStokastik:
    def __init__(self, daftar_parameter, laju_adaptasi):
        self.daftar_parameter = daftar_parameter
        self.laju_adaptasi = laju_adaptasi

    def reset_gradien_ke_nol(self):
        for elemen_parameter in self.daftar_parameter:
            elemen_parameter.gradien.data = np.multiply(elemen_parameter.gradien.data, 0.0)

    def eksekusi_langkah_koreksi(self):
        for elemen_parameter in self.daftar_parameter:
            potongan_revisi = np.multiply(elemen_parameter.gradien.data, self.laju_adaptasi)
            elemen_parameter.data = np.subtract(elemen_parameter.data, potongan_revisi)

print("Modul pengoptimalan cerdas siap beroperasi menyisir medan data operasional")

Modul pengoptimalan cerdas siap beroperasi menyisir medan data operasional


### Pengujian Terpadu Arsitektur Jaringan Saraf Masa Depan
Sekarang tibalah saatnya untuk membuktikan elegansi kerangka kerja buatan tangan kita. Kita menyiapkan sebuah kasus prediksi linear lazim. Anda akan melihat secara langsung bahwa seluruh siklus penulisan algoritma pembelajaran kini menjadi sangat ringkas, bersih, dan mempesona. Kita hanya perlu menginstruksikan perkalian matriks, memicu kalkulasi kerugian kuadrat, memberikan komando pemicu perambatan mundur secara sekejap, lalu mendelegasikan wewenang pembaruan sistem kepada sang pengoptimal.

In [4]:
fakta_lapangan = KerangkaTensor(np.array([[0.0, 1.0], [1.0, 0.0], [1.0, 1.0], [0.0, 0.0]]), optimasi_otomatis=True)
harapan_pasti = KerangkaTensor(np.array([[1.0], [1.0], [0.0], [0.0]]), optimasi_otomatis=True)

np.random.seed(0)
parameter_jaringan = KerangkaTensor(np.random.rand(2, 1), optimasi_otomatis=True)

modul_pengawas = PenurunanGradienStokastik(daftar_parameter=[parameter_jaringan], laju_adaptasi=0.1)

for siklus_pelatihan in range(25):
    tebakan_jaringan = fakta_lapangan.eksekusi_dot(parameter_jaringan)

    jarak_deviasi = tebakan_jaringan.__sub__(harapan_pasti)
    kerugian_kuadrat = jarak_deviasi * jarak_deviasi

    kerugian_kuadrat.mundur()

    evaluasi_agregat = np.sum(kerugian_kuadrat.data)

    modul_pengawas.eksekusi_langkah_koreksi()
    modul_pengawas.reset_gradien_ke_nol()

    if (np.add(siklus_pelatihan, 1) % 5) == 0:
        catatan_putaran = np.add(siklus_pelatihan, 1)
        print("Kondisi kecerdasan pada putaran", catatan_putaran, "merekam reduksi kerugian di ambang batas", evaluasi_agregat)

print("\nSistem kerangka kerja modern sukses menunjukkan supremasi otonomi diferensiasi tanpa campur tangan kalkulus pihak luar.")

Kondisi kecerdasan pada putaran 5 merekam reduksi kerugian di ambang batas 1.3360061368742355
Kondisi kecerdasan pada putaran 10 merekam reduksi kerugian di ambang batas 1.333582697743572
Kondisi kecerdasan pada putaran 15 merekam reduksi kerugian di ambang batas 1.3333601046876633
Kondisi kecerdasan pada putaran 20 merekam reduksi kerugian di ambang batas 1.3333362078852025
Kondisi kecerdasan pada putaran 25 merekam reduksi kerugian di ambang batas 1.33333364198599

Sistem kerangka kerja modern sukses menunjukkan supremasi otonomi diferensiasi tanpa campur tangan kalkulus pihak luar.
